# EDA - Olist Delivery


In [1]:
import sys
from pathlib import Path

# Find the project root by walking up until src/olist is visible, so the
# notebook runs from any working directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "olist").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from olist import config as cfg, io_utils as io, plots

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
cfg.ensure_dirs()

clean = io.load_parquet_dir(cfg.CLEAN_DIR)
feat = io.load_parquet_dir(cfg.FEATURES_DIR)

orders = clean["olist_orders_dataset"]
items = clean["olist_order_items_dataset"]
customers = clean["olist_customers_dataset"]
products = clean["olist_products_dataset"]

order_features = feat["order_features"]
customer_rfm = feat["customer_features"]
product_features = feat["product_features"]
seller_features = feat["seller_features"]
review_features = feat["review_features"]
order_geo = feat["order_geo_features"]

pct = plots.pct
print("order_features:", order_features.shape)

order_features: (99441, 61)


In [2]:
def save_fig(fig, question_no, slug, insight):
    path = plots.stamp_and_save(fig, plots.eda_figure_path(question_no, slug), insight)
    plt.show()
    print(f"saved -> {path.relative_to(ROOT)}")
    return path


order_features["purchase_month_period"] = pd.PeriodIndex(order_features["purchase_month"], freq="M")
DATA_CUTOFF = order_features["Order_Purchase_Timestamp"].max()
CUTOFF_MONTH = DATA_CUTOFF.to_period("M")
complete = order_features[order_features["purchase_month_period"] < CUTOFF_MONTH]

print("data cutoff  :", DATA_CUTOFF)
print("cutoff month :", CUTOFF_MONTH, "(excluded from trends - incomplete)")
print("complete rows:", f"{len(complete):,} of {len(order_features):,}")

data cutoff  : 2018-10-17 17:30:18
cutoff month : 2018-10 (excluded from trends - incomplete)
complete rows: 99,437 of 99,441


## 1. Delivery performance over time


In [3]:
monthly = (
    complete.groupby("purchase_month_period")
    .agg(orders=("Order_Id", "nunique"), gmv=("order_gmv", "sum"), aov=("order_gmv", "mean"))
    .reset_index()
)
monthly["month"] = monthly["purchase_month_period"].astype(str)

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.bar(monthly["month"], monthly["orders"], color="steelblue", alpha=0.75, label="Orders")
ax1.set_ylabel("Orders")
ax1.tick_params(axis="x", rotation=90)
ax2 = ax1.twinx()
ax2.plot(monthly["month"], monthly["gmv"], color="darkorange", marker="o", label="GMV proxy")
ax2.set_ylabel("GMV proxy (BRL)")
ax1.set_title("Monthly orders and GMV proxy")

first, last = monthly.iloc[0], monthly.iloc[-1]
peak = monthly.loc[monthly["gmv"].idxmax()]
insight = (
    f"INSIGHT: orders moved {pct(last['orders'] / first['orders'] - 1, 0)} between {first['month']} and "
    f"{last['month']} ({first['orders']:,.0f} -> {last['orders']:,.0f}) while AOV moved only "
    f"{pct(last['aov'] / first['aov'] - 1, 0)} ({first['aov']:,.0f} -> {last['aov']:,.0f} BRL) "
    f"=> growth is volume-driven, not basket-driven. Peak GMV month: {peak['month']} "
    f"({peak['gmv']:,.0f} BRL). Cutoff month {CUTOFF_MONTH} excluded as incomplete."
)
save_fig(fig, 1, "monthly_trend_orders_gmv_aov", insight)

saved -> reports\figures\q01_monthly_trend_orders_gmv_aov.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q01_monthly_trend_orders_gmv_aov.png')

## 2. Pareto setup


In [4]:
TOP_N, PARETO = 15, 80
cat_revenue = (
    product_features.groupby("Product_Category_Name_English", dropna=True)["product_revenue"]
    .sum().sort_values(ascending=False)
)
cat_revenue = cat_revenue[cat_revenue.index.notna()]
cum_share = cat_revenue.cumsum() / cat_revenue.sum() * 100
show_n = min(TOP_N, len(cat_revenue))

fig, ax1 = plt.subplots(figsize=(12, 6))
cat_revenue.head(show_n).plot(kind="bar", ax=ax1, color="teal")
ax1.set_ylabel("GMV proxy (BRL)")
ax1.set_xlabel("")
ax1.tick_params(axis="x", rotation=75)
ax2 = ax1.twinx()
ax2.plot(range(show_n), cum_share.head(show_n).values, color="red", marker="o")
ax2.axhline(PARETO, color="gray", ls="--")
ax2.set_ylabel("Cumulative share (%)")
ax2.set_ylim(0, 100)
ax1.set_title(f"Category Pareto (top {show_n} of {len(cat_revenue)})")

n_pareto = int((cum_share <= PARETO).sum()) + 1
top3 = cat_revenue.head(3)
insight = (
    f"INSIGHT: {n_pareto} of {len(cat_revenue)} categories ({n_pareto / len(cat_revenue):.0%}) produce "
    f"{PARETO}% of GMV. Top 3 ({', '.join(top3.index[:3])}) alone are "
    f"{top3.sum() / cat_revenue.sum():.1%}. The remaining {len(cat_revenue) - n_pareto} are long tail: "
    f"assortment decisions there move almost nothing."
)
save_fig(fig, 2, "category_gmv_pareto", insight)

saved -> reports\figures\q02_category_gmv_pareto.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q02_category_gmv_pareto.png')

## 3. Order funnel


In [5]:
funnel = pd.Series({
    "Purchased": len(order_features),
    "Approved": int(order_features["Order_Approved_At"].notna().sum()),
    "Handed to carrier": int(order_features["Order_Delivered_Carrier_Date"].notna().sum()),
    "Delivered": int(order_features["Order_Delivered_Customer_Date"].notna().sum()),
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
funnel.plot(kind="bar", ax=axes[0], color="slateblue")
axes[0].set_title("Order delivery funnel")
axes[0].tick_params(axis="x", rotation=20)
for i, v in enumerate(funnel.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

delay = order_features["delivery_delay_days"].dropna()
sns.histplot(delay.clip(-30, 30), bins=60, ax=axes[1], color="crimson")
axes[1].axvline(0, color="black", ls="--")
axes[1].set_title("Delay vs promise (days). Negative = early")
axes[1].set_xlabel("delivery_delay_days")

late_rate = order_features["late_delivery_flag"].mean()
insight = (
    f"INSIGHT: {pct(funnel['Delivered'] / funnel['Purchased'])} of orders reach the customer and "
    f"{pct(late_rate)} of those arrive late. The median order lands "
    f"{abs(delay.median()):.0f} days {'early' if delay.median() < 0 else 'late'}, so the promise is "
    f"heavily padded and the problem is the tail: p95 delay is {delay.quantile(0.95):.0f} days and "
    f"{pct((delay > 7).mean())} of orders are more than a week late."
)
save_fig(fig, 3, "delivery_funnel_and_delay", insight)

saved -> reports\figures\q03_delivery_funnel_and_delay.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q03_delivery_funnel_and_delay.png')

## 4. Category focus


In [6]:
MIN_ORDERS = 50
state_late = (
    order_features.groupby("Customer_State")
    .agg(orders=("Order_Id", "nunique"), late_rate=("late_delivery_flag", "mean"))
    .query("orders >= @MIN_ORDERS").sort_values("late_rate", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=state_late["late_rate"].head(15) * 100, y=state_late.head(15).index, ax=ax, color="orangered")
national = order_features["late_delivery_flag"].mean()
ax.axvline(national * 100, color="black", ls="--", label=f"national {pct(national)}")
ax.set(xlabel="Late-delivery rate (%)", ylabel="Customer state",
       title=f"Worst states by late rate (min {MIN_ORDERS} orders)")
ax.legend()

worst, best = state_late.iloc[0], state_late.iloc[-1]
insight = (
    f"INSIGHT: {worst.name} is worst at {pct(worst['late_rate'])} ({worst['orders']:,.0f} orders), "
    f"{worst['late_rate'] / max(best['late_rate'], 1e-9):.1f}x the best qualifying state {best.name} "
    f"({pct(best['late_rate'])}). National average {pct(national)}; "
    f"{int((state_late['late_rate'] > national).sum())} of {len(state_late)} states sit above it "
    f"=> lateness is a regional capacity problem, not a platform-wide one."
)
save_fig(fig, 4, "late_rate_by_state", insight)

saved -> reports\figures\q04_late_rate_by_state.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q04_late_rate_by_state.png')

## 5. Delivery segments


In [7]:
seg = (
    customer_rfm.groupby("rfm_segment_label")
    .agg(customers=("Customer_Unique_Id", "size"), total_value=("monetary_value", "sum"),
         avg_recency=("recency_days", "mean"), avg_frequency=("frequency_orders", "mean"))
    .sort_values("total_value", ascending=False)
)
seg["customer_share_pct"] = (seg["customers"] / seg["customers"].sum() * 100).round(1)
seg["value_share_pct"] = (seg["total_value"] / seg["total_value"].sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
seg["customers"].plot(kind="bar", ax=axes[0], color="mediumseagreen")
axes[0].set(title="Customers per RFM segment", xlabel="")
axes[0].tick_params(axis="x", rotation=20)
seg["value_share_pct"].plot(kind="bar", ax=axes[1], color="goldenrod")
axes[1].set(title="Share of GMV proxy per segment (%)", xlabel="")
axes[1].tick_params(axis="x", rotation=20)

top = seg.iloc[0]
one_time = customer_rfm["frequency_orders"].eq(1).mean()
insight = (
    f"INSIGHT: '{top.name}' holds {top['value_share_pct']:.1f}% of value on "
    f"{top['customer_share_pct']:.1f}% of customers "
    f"({top['value_share_pct'] / max(top['customer_share_pct'], 1e-9):.1f}x leverage). But "
    f"{pct(one_time)} of customers bought exactly once, so F and M carry almost no variance: treat "
    f"this as a diagnostic, not a retention programme. Repeat behaviour is the scarce asset here."
)
save_fig(fig, 5, "rfm_segments", insight)

saved -> reports\figures\q05_rfm_segments.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q05_rfm_segments.png')

## 6. Late delivery flag


In [8]:
rd = review_features[review_features["late_delivery_flag"].notna()].copy()
rd["delivery_bucket"] = np.where(rd["late_delivery_flag"] == 1, "Late", "On-time/Early")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=rd, x="delivery_bucket", y="Review_Score", ax=axes[0],
            order=["Late", "On-time/Early"], palette=["#e76f51", "#2a9d8f"], hue="delivery_bucket",
            legend=False)
axes[0].set(xlabel="", title="Review score by delivery status")

buckets = pd.cut(rd["delivery_delay_days"], [-np.inf, -10, -5, 0, 5, 10, np.inf],
                 labels=["10+ early", "5-10 early", "0-5 early", "0-5 late", "5-10 late", "10+ late"])
by_bucket = rd.groupby(buckets, observed=True)["Review_Score"].agg(["mean", "size"])
axes[1].bar(by_bucket.index.astype(str), by_bucket["mean"], color="#264653")
axes[1].set(ylim=(1, 5), ylabel="Mean review score", title="Score decays with delay")
axes[1].tick_params(axis="x", rotation=25)

means = rd.groupby("delivery_bucket")["Review_Score"].mean()
bad = rd.assign(bad=rd["Review_Score"].le(2)).groupby("delivery_bucket")["bad"].mean()
late_m, ontime_m = means.get("Late", np.nan), means.get("On-time/Early", np.nan)
insight = (
    f"INSIGHT: late orders average {late_m:.2f}/5 vs {ontime_m:.2f}/5 on time (a {ontime_m - late_m:.2f} "
    f"star gap). {pct(bad.get('Late', np.nan))} of late orders get 1-2 stars vs "
    f"{pct(bad.get('On-time/Early', np.nan))} when on time "
    f"({bad.get('Late', np.nan) / max(bad.get('On-time/Early', 1e-9), 1e-9):.1f}x), and the decay is "
    f"monotonic in delay size => delivery timing is the dominant observable driver of satisfaction."
)
save_fig(fig, 6, "review_score_vs_delay", insight)

saved -> reports\figures\q06_review_score_vs_delay.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q06_review_score_vs_delay.png')

## 7. Review focus


In [9]:
MIN_REVIEWS = 30
review_product = (
    review_features.merge(items[["Order_Id", "Product_Id"]].drop_duplicates("Order_Id"),
                          on="Order_Id", how="left")
    .merge(products[["Product_Id", "Product_Category_Name_English"]], on="Product_Id", how="left")
)
cat_review = (
    review_product.groupby("Product_Category_Name_English")
    .agg(reviews=("Review_Id", "count"), avg_score=("Review_Score", "mean"))
    .query("reviews >= @MIN_REVIEWS").sort_values("avg_score")
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=cat_review["avg_score"].head(15), y=cat_review.head(15).index, ax=ax, color="firebrick")
overall = review_product["Review_Score"].mean()
ax.axvline(overall, color="black", ls="--", label=f"platform mean {overall:.2f}")
ax.set(xlim=(1, 5), xlabel="Average review score", ylabel="",
       title=f"Worst categories by review score (min {MIN_REVIEWS} reviews)")
ax.legend()

worst = cat_review.iloc[0]
insight = (
    f"INSIGHT: platform mean is {overall:.2f}/5. Worst category '{worst.name}' scores "
    f"{worst['avg_score']:.2f} over {worst['reviews']:,.0f} reviews, "
    f"{overall - worst['avg_score']:.2f} stars below. "
    f"{int((cat_review['avg_score'] < overall).sum())} of {len(cat_review)} qualifying categories sit "
    f"below the mean - but check Q6 first: much of this is delivery, not product quality."
)
save_fig(fig, 7, "worst_review_categories", insight)

saved -> reports\figures\q07_worst_review_categories.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q07_worst_review_categories.png')

## 8. Sentiment analysis


In [10]:
by_sentiment = review_features.groupby("sentiment_label").agg(
    reviews=("Review_Id", "count"), avg_length=("review_length", "mean"),
    avg_words=("review_word_count", "mean"), has_text_rate=("Has_Review_Text", "mean"),
)
order_ = [s for s in ["negative", "neutral", "positive"] if s in by_sentiment.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(order_, by_sentiment.loc[order_, "avg_words"], color=["#e63946", "#f4a261", "#2a9d8f"][:len(order_)])
axes[0].set(ylabel="Average word count", title="Review length by sentiment")
axes[1].bar(order_, by_sentiment.loc[order_, "has_text_rate"] * 100,
            color=["#e63946", "#f4a261", "#2a9d8f"][:len(order_)])
axes[1].set(ylabel="% leaving free text", title="Willingness to explain")

neg, pos = by_sentiment.loc["negative"], by_sentiment.loc["positive"]
insight = (
    f"INSIGHT: negative reviews average {neg['avg_words']:.1f} words vs {pos['avg_words']:.1f} for "
    f"positive ({neg['avg_words'] / max(pos['avg_words'], 1e-9):.1f}x), and "
    f"{pct(neg['has_text_rate'])} of them carry free text vs {pct(pos['has_text_rate'])} "
    f"=> unhappy customers explain themselves. The {neg['reviews']:,.0f} negative reviews are where the "
    f"root-cause signal lives, and are the obvious next dataset for a text model (out of scope here)."
)
save_fig(fig, 8, "review_effort_by_sentiment", insight)

saved -> reports\figures\q08_review_effort_by_sentiment.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q08_review_effort_by_sentiment.png')

## 9. Delivery time distribution


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
share = order_features["dominant_payment_type"].value_counts(normalize=True) * 100
share.plot(kind="bar", ax=axes[0], color="cornflowerblue")
axes[0].set(title="Dominant payment method (% of orders)", xlabel="")
axes[0].tick_params(axis="x", rotation=30)

inst = (
    order_features[order_features["max_installments"].between(1, 12)]
    .groupby("max_installments")["order_gmv"].median()
)
inst.plot(marker="o", ax=axes[1], color="darkgreen")
axes[1].set(title="Median order value by installment count", xlabel="max_installments",
            ylabel="Median GMV (BRL)")

v1, vmax_i = inst.loc[1], inst.index.max()
insight = (
    f"INSIGHT: {share.index[0]} dominates at {share.iloc[0]:.1f}% of orders "
    f"({share.index[1]} second at {share.iloc[1]:.1f}%). Median order value rises from {v1:,.0f} BRL at "
    f"1 installment to {inst.loc[vmax_i]:,.0f} at {vmax_i:.0f} "
    f"({inst.loc[vmax_i] / max(v1, 1e-9):.1f}x) => installments are how customers finance big baskets, "
    f"so installment availability is a basket-size lever, not just a payments detail."
)
save_fig(fig, 9, "payment_and_installments", insight)

saved -> reports\figures\q09_payment_and_installments.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q09_payment_and_installments.png')

## 10. Seller revenue


In [12]:
srt = seller_features.sort_values("seller_revenue", ascending=False).reset_index(drop=True)
total = srt["seller_revenue"].sum()
srt["cum_share"] = srt["seller_revenue"].cumsum() / total * 100
top10n = max(1, int(len(srt) * 0.1))
top10share = srt.loc[: top10n - 1, "seller_revenue"].sum() / total * 100
n80 = int((srt["cum_share"] <= 80).sum()) + 1

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, len(srt) + 1), srt["cum_share"], color="purple")
ax.axvline(top10n, color="gray", ls="--", label=f"top 10% = {top10n:,} sellers")
ax.axhline(80, color="orange", ls=":", label="80% of GMV")
ax.set(xlabel="Sellers, ranked by GMV", ylabel="Cumulative GMV share (%)",
       title="Seller GMV concentration")
ax.legend()

insight = (
    f"INSIGHT: the top 10% of sellers ({top10n:,} of {len(srt):,}) generate {top10share:.1f}% of GMV, "
    f"and just {n80:,} sellers ({n80 / len(srt):.1%}) cover 80%. That is a concrete churn-dependency "
    f"risk: losing a two-digit number of sellers would be a revenue event, so seller retention deserves "
    f"the same instrumentation as customer retention."
)
save_fig(fig, 10, "seller_gmv_concentration", insight)

saved -> reports\figures\q10_seller_gmv_concentration.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q10_seller_gmv_concentration.png')

## 11. Payment types


In [13]:
by_pay = order_features.groupby("dominant_payment_type")["cancelled_flag"].mean().sort_values(ascending=False)
by_items = order_features.groupby(
    pd.cut(order_features["order_item_count"], [0, 1, 2, 3, 100], labels=["1", "2", "3", "4+"]),
    observed=True,
)["cancelled_flag"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
by_pay.plot(kind="bar", ax=axes[0], color="salmon")
axes[0].set(title="Cancellation rate by payment method", xlabel="")
axes[0].tick_params(axis="x", rotation=30)
by_items.plot(kind="bar", ax=axes[1], color="indianred")
axes[1].set(title="Cancellation rate by items per order", xlabel="Items")

overall = order_features["cancelled_flag"].mean()
insight = (
    f"INSIGHT: overall cancellation is {pct(overall, 2)}, but '{by_pay.index[0]}' runs at "
    f"{pct(by_pay.iloc[0], 2)} ({by_pay.iloc[0] / max(overall, 1e-9):.1f}x baseline) and baskets with "
    f"{by_items.idxmax()} item(s) cancel most at {pct(by_items.max(), 2)} => cancellation is a payment "
    f"confirmation problem, not a volume problem. At this base rate it is a monitoring metric, not a "
    f"modelling target: there is not enough signal to predict it usefully."
)
save_fig(fig, 11, "cancellation_drivers", insight)

saved -> reports\figures\q11_cancellation_drivers.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q11_cancellation_drivers.png')

## 12. Distance and lateness


In [14]:
gd = order_features.dropna(subset=["distance_km", "late_delivery_flag"])
dist_late = gd.groupby("distance_bucket", observed=True).agg(
    orders=("Order_Id", "size"), late_rate=("late_delivery_flag", "mean"),
    avg_delivery_days=("delivery_days", "mean"),
    avg_promise_days=("estimated_delivery_days", "mean"),
).reindex(["<50km", "50-200km", "200-500km", "500-1000km", "1000km+"]).dropna(how="all")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(dist_late.index, dist_late["late_rate"] * 100, color="teal")
axes[0].set(ylabel="Late rate (%)", title="Late rate by shipping distance")
axes[0].tick_params(axis="x", rotation=15)

x = np.arange(len(dist_late))
axes[1].bar(x - 0.2, dist_late["avg_promise_days"], 0.4, label="promised", color="#adb5bd")
axes[1].bar(x + 0.2, dist_late["avg_delivery_days"], 0.4, label="actual", color="#e76f51")
axes[1].set_xticks(x)
axes[1].set_xticklabels(dist_late.index, rotation=15)
axes[1].set(ylabel="Days", title="Promise barely scales with distance; actual does")
axes[1].legend()

near, far = dist_late["late_rate"].iloc[0], dist_late["late_rate"].iloc[-1]
promise_growth = dist_late["avg_promise_days"].iloc[-1] / dist_late["avg_promise_days"].iloc[0]
actual_growth = dist_late["avg_delivery_days"].iloc[-1] / dist_late["avg_delivery_days"].iloc[0]
insight = (
    f"INSIGHT: late rate climbs from {pct(near)} under 50km to {pct(far)} beyond 1000km "
    f"({far / max(near, 1e-9):.1f}x). Actual transit stretches {actual_growth:.1f}x across those bands "
    f"while the promise stretches only {promise_growth:.1f}x => the promise is under-indexed on distance. "
    f"Scaling the promised date with distance is the cheapest fix available and needs no model."
)
save_fig(fig, 12, "late_rate_by_distance", insight)

saved -> reports\figures\q12_late_rate_by_distance.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q12_late_rate_by_distance.png')

## 13. Seller reliability


In [15]:
MIN_SELLER_ORDERS = 30
sc = seller_features[seller_features["seller_order_count"] >= MIN_SELLER_ORDERS].copy()
national = order_features["late_delivery_flag"].mean()
sc["excess_late_orders"] = (sc["seller_late_rate"] - national) * sc["seller_order_count"]
worst = sc.nlargest(15, "excess_late_orders")

fig, ax = plt.subplots(figsize=(11, 6))
scat = ax.scatter(sc["seller_order_count"], sc["seller_late_rate"] * 100,
                  s=sc["seller_revenue"] / sc["seller_revenue"].max() * 400 + 15,
                  c=sc["seller_late_rate"] * 100, cmap="RdYlGn_r", alpha=0.75, edgecolor="grey")
ax.axhline(national * 100, color="black", ls="--", label=f"national {pct(national)}")
ax.set(xscale="log", xlabel="Orders shipped (log)", ylabel="Late rate (%)",
       title=f"Seller reliability vs volume (bubble = GMV, min {MIN_SELLER_ORDERS} orders)")
ax.legend()
fig.colorbar(scat, ax=ax, label="Late rate (%)")

insight = (
    f"INSIGHT: of {len(sc):,} sellers with {MIN_SELLER_ORDERS}+ orders, "
    f"{int((sc['seller_late_rate'] > national).sum())} sit above the national late rate. The worst 15 by "
    f"*excess late orders* account for {worst['excess_late_orders'].sum():,.0f} avoidable late deliveries "
    f"on {worst['seller_revenue'].sum() / seller_features['seller_revenue'].sum():.1%} of GMV => ranking "
    f"by excess volume, not by rate, is what makes this an actionable call list rather than a leaderboard "
    f"of tiny sellers with unlucky samples."
)
save_fig(fig, 13, "seller_reliability_vs_volume", insight)
print(worst[["Seller_Id", "Seller_State", "seller_order_count", "seller_late_rate",
             "excess_late_orders"]].round(3).to_string(index=False))

saved -> reports\figures\q13_seller_reliability_vs_volume.png
                       Seller_Id Seller_State  seller_order_count  seller_late_rate  excess_late_orders
06a2c3af7b3aee5d69171b0e14f0ee87           MA                 392             0.236              60.834
4a3ca9315b744ce9f8e9374361493884           SP                1806             0.110              51.780
4869f7a5dfa277a7dca6462dcf3b52b2           SP                1132             0.116              39.308
8160255418d5aaa7dbdc9f4c64ebda44           SP                 385             0.179              37.775
88460e8ebdecbfecb5f9601833981930           PR                 248             0.197              28.653
ea8482cd71df3c1969d7b9473ff13abc           SP                1146             0.104              25.678
e5a3438891c0bfdb9394643f95273d8e           SP                 224             0.182              22.637
7d13fca15225358621be4086e1eb0964           SP                 565             0.119              21.448
8b

C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. State demand


In [16]:
state_demand = (
    order_features.groupby("Customer_State")
    .agg(orders=("Order_Id", "nunique"), gmv=("order_gmv", "sum"),
         late_rate=("late_delivery_flag", "mean"), avg_distance=("distance_km", "mean"))
    .sort_values("orders", ascending=False)
)

top_n = min(15, len(state_demand))
top = state_demand.head(top_n)

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(x=top["orders"], y=top.index, color="cadetblue", ax=ax)
ax.set(xlabel="Orders", ylabel="Customer state", title=f"Top {top_n} states by order volume")
ax2 = ax.twiny()
ax2.plot(top["late_rate"] * 100, range(top_n), "o--", color="orangered", label="late rate")
ax2.set_xlabel("Late rate (%)")
ax2.legend(loc="lower right")

total_orders = state_demand["orders"].sum()
top1 = state_demand.iloc[0]
insight = (
    f"INSIGHT: {top1.name} alone takes {top1['orders'] / total_orders:.1%} of orders and "
    f"{top1['gmv'] / state_demand['gmv'].sum():.1%} of GMV at a {pct(top1['late_rate'])} late rate; the "
    f"top 5 states cover {pct(state_demand['orders'].head(5).sum() / total_orders)} of volume across "
    f"{len(state_demand)} states. Demand and reliability move together because the seller base is "
    f"concentrated in the southeast => the long tail is a fulfilment-network problem, not a demand problem."
)
save_fig(fig, 14, "demand_by_state", insight)

saved -> reports\figures\q14_demand_by_state.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q14_demand_by_state.png')

## 15. Cost-like features


In [17]:
cost_like = [c for c in order_features.columns
             if any(k in c.lower() for k in ("cost", "margin", "profit", "cogs", "commission"))]
gmv_total = order_features["order_gmv"].sum()
freight_total = order_features["order_freight_total"].sum()

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(["GMV proxy"], [gmv_total], color="#2a9d8f")
ax.bar(["Cost structure"], [freight_total], color="#e76f51", label="Freight (observable)")
ax.bar(["Cost structure"], [gmv_total - freight_total], bottom=[freight_total], color="#cccccc",
       hatch="//", label="COGS + commission + marketing (NOT in dataset)")
ax.set(ylabel="BRL", title="What we can measure vs what profit would require")
ax.legend(loc="upper right", fontsize=9)

insight = (
    f"INSIGHT: freight is the only observable cost, at {freight_total:,.0f} BRL "
    f"({freight_total / gmv_total:.1%} of the {gmv_total:,.0f} BRL GMV proxy). The dataset contains "
    f"{len(cost_like)} cost or margin columns => any 'profit' or 'ROI' number from this data would be a "
    f"fabrication. This project reports GMV, freight and delivery outcomes, and states its cost "
    f"assumptions explicitly wherever a money figure appears in phase 4."
)
save_fig(fig, 15, "gmv_vs_profit_limitation", insight)

saved -> reports\figures\q15_gmv_vs_profit_limitation.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


WindowsPath('C:/Users/programmer/Desktop/program/ali-ajvand-data-portfolio/olist-delivery-analytics/reports/figures/q15_gmv_vs_profit_limitation.png')

## 16. Measures


In [18]:
measures = {
    "orders_total": order_features["Order_Id"].nunique(),
    "orders_delivered": int(order_features["delivered_flag"].sum()),
    "gmv_proxy_brl": round(float(order_features["order_gmv"].sum()), 2),
    "freight_total_brl": round(float(order_features["order_freight_total"].sum()), 2),
    "payments_total_brl": round(float(order_features["order_total_paid"].sum()), 2),
    "items_total": int(order_features["order_item_count"].sum()),
    "aov_brl": round(float(order_features["order_gmv"].mean()), 2),
    "unique_customers": int(order_features["Customer_Unique_Id"].nunique()),
    "unique_sellers": int(seller_features["Seller_Id"].nunique()),
    "late_delivery_rate": round(float(order_features["late_delivery_flag"].mean()), 6),
    "cancellation_rate": round(float(order_features["cancelled_flag"].mean()), 6),
    "avg_delivery_days": round(float(order_features["delivery_days"].mean()), 4),
    "median_delay_days": round(float(order_features["delivery_delay_days"].median()), 4),
    "avg_review_score": round(float(review_features["Review_Score"].mean()), 4),
    "bad_review_rate": round(float(review_features["is_bad_review"].mean()), 6),
    "avg_distance_km": round(float(order_features["distance_km"].mean()), 2),
    "repeat_customer_rate": round(float(customer_rfm["repeat_customer_flag"].mean()), 6),
}
validation = pd.DataFrame(measures.items(), columns=["measure", "python_value"])
validation["bi_value"] = pd.NA
validation["match"] = pd.NA
out = cfg.REPORTS_DIR / "bi_validation_aggregates.csv"
validation.to_csv(out, index=False)
print(f"wrote {out.relative_to(ROOT)}\n")
validation

wrote reports\bi_validation_aggregates.csv



,measure,python_value,bi_value,match
0,orders_total,9.944100e+04,<NA>,<NA>
1,orders_delivered,9.647600e+04,<NA>,<NA>
2,gmv_proxy_brl,1.359164e+07,<NA>,<NA>
3,freight_total_brl,2.251910e+06,<NA>,<NA>
4,payments_total_brl,1.600887e+07,<NA>,<NA>
5,items_total,1.126500e+05,<NA>,<NA>
6,aov_brl,1.377500e+02,<NA>,<NA>
7,unique_customers,9.609600e+04,<NA>,<NA>
8,unique_sellers,3.095000e+03,<NA>,<NA>
9,late_delivery_rate,8.112900e-02,<NA>,<NA>


In [19]:
# Grain-level reconciliation: raw tables vs feature tables. Any drift means a
# join fanned out, and every BI measure built on top would inherit the error.
recon = pd.DataFrame([
    ("unique orders", orders["Order_Id"].nunique(), order_features["Order_Id"].nunique()),
    ("item price total", items["Price"].sum(), order_features["order_gmv"].sum()),
    ("freight total", items["Freight_Value"].sum(), order_features["order_freight_total"].sum()),
    ("item rows", len(items), order_features["order_item_count"].sum()),
], columns=["metric", "clean_tables", "feature_tables"])
recon["diff"] = recon["feature_tables"] - recon["clean_tables"]
recon["diff_pct"] = (recon["diff"] / recon["clean_tables"] * 100).round(5)

fig, ax = plt.subplots(figsize=(11, 1.2 + 0.5 * len(recon)))
ax.axis("off")
table = ax.table(
    cellText=[[r["metric"], f"{r['clean_tables']:,.2f}", f"{r['feature_tables']:,.2f}",
               f"{r['diff']:,.2f}", f"{r['diff_pct']:.4f}%"] for _, r in recon.iterrows()],
    colLabels=["Metric", "Clean tables", "Feature tables", "Diff", "Diff %"],
    cellLoc="center", loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.6)
ax.set_title("Clean vs feature-table reconciliation", pad=16)

max_diff = recon["diff_pct"].abs().max()
status = "PASS" if max_diff < 0.01 else "REVIEW"
insight = (
    f"INSIGHT [{status}]: the largest gap between clean and feature tables is {max_diff:.4f}% across "
    f"{len(recon)} metrics. "
    + ("Joins preserved grain, so the BI semantic model can be built directly on the feature tables."
       if status == "PASS" else "A join duplicated rows - fix it before building any dashboard.")
)
save_fig(fig, 16, "bi_reconciliation", insight)
assert max_diff < 0.01

saved -> reports\figures\q16_bi_reconciliation.png


C:\Users\programmer\AppData\Local\Temp\ipykernel_25300\869683805.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 17. Summary
